# 03 · The same advisor as an ADK agent
**Time:** 20 min · **Goal:** build the agent with Google's open-source Agent Development Kit (ADK).

Run the cells from top to bottom. **Do not share your key and do not upload private data.** Free-tier content may be used by Google to improve its products.

Get a key: https://aistudio.google.com/apikey

> ADK 2.0 changed parts of its API. If a cell fails, compare with the docs: https://google.github.io/adk-docs and the workshop repo https://github.com/saurabh1611a7458-pixel/Agent-Voice

In [ ]:
# Setup: install the Gemini SDK and load your free AI Studio key
!pip -q install google-genai
import os, getpass
try:
    from google.colab import userdata          # Colab: left menu > Secrets > add GOOGLE_API_KEY
    API_KEY = userdata.get("GOOGLE_API_KEY")
except Exception:
    API_KEY = os.environ.get("GOOGLE_API_KEY") or getpass.getpass("Paste your AI Studio key: ")
from google import genai
from google.genai import types
client = genai.Client(api_key=API_KEY)
# Model names retire quickly. This tries several free-tier models and uses the first that works.
CANDIDATE_MODELS = ["gemini-3.5-flash-lite", "gemini-3.1-flash-lite", "gemini-3.5-flash"]
def pick_model():
    for m in CANDIDATE_MODELS:
        try:
            client.models.generate_content(model=m, contents="ping")
            print("Using model:", m)
            return m
        except Exception as e:
            print("Skipping", m, "->", str(e)[:100])
    raise RuntimeError("No model worked. Check your key, account and the model list at https://ai.google.dev/gemini-api/docs/models")
MODEL = pick_model()
!pip -q install google-adk
os.environ["GOOGLE_API_KEY"] = API_KEY

In [ ]:
from google.adk import Agent
from google.adk.runners import InMemoryRunner

CROP_TIPS = {
    ("maize", "yellow leaves"): "Check nitrogen, drainage and stalk borer.",
    ("maize", "holes in leaves"): "Possible fall armyworm. Inspect early in the morning and contact the extension officer.",
}

def crop_advice(crop: str, problem: str) -> dict:
    """Look up advice for a crop problem. crop and problem are short English phrases."""
    return {"advice": CROP_TIPS.get((crop.lower().strip(), problem.lower().strip()), "No record. Ask the extension officer.")}

root_agent = Agent(
    name="shamba_agent",
    model=MODEL,
    instruction=("Advise smallholder farmers briefly, in the farmer's language (Swahili by default). "
                 "Use crop_advice for crop problems. If unsure, say so."),
    tools=[crop_advice],
)

## Run it
A runner keeps the session. `await` works directly in Colab cells.

In [ ]:
runner = InMemoryRunner(agent=root_agent, app_name="shamba")
session = await runner.session_service.create_session(app_name="shamba", user_id="farmer1")

async def ask(text):
    msg = types.Content(role="user", parts=[types.Part(text=text)])
    async for event in runner.run_async(user_id="farmer1", session_id=session.id, new_message=msg):
        if event.is_final_response() and event.content and event.content.parts:
            print("Agent:", event.content.parts[0].text)

await ask("Majani ya mahindi yangu yanageuka manjano. Nifanye nini?")
await ask("Na nikiona mashimo kwenye majani?")

## Your turn
- Change the instruction to refuse questions that are not about farming.
- Add a second tool, for example `market_price(crop)` that returns a made-up price table.
- Locally, you can also run the built-in test UI with `adk web` (see the README).